# 02 — Intent classification benchmark

**Phase B** (`docs/roadmap.md`). The comparable baseline the drift paper rests
on: intent classification on Ninapro DB6, scored under **four evaluation
protocols** rather than one. Feeds `reborn.ml.intent.IntentClassifier` once a
model is worth wiring in.

| Protocol | Train | Test | What it measures |
|---|---|---|---|
| Within-session | part of session *k* | rest of session *k* | Ceiling |
| **Cross-session** | earlier sessions | a later session | **Drift** |
| Cross-subject | other subjects | held-out subject | Worst case |
| Random-shuffle | shuffled windows | shuffled windows | **Control, not a result** |

**On the random-shuffle control.** Windows overlap by 150 ms at a 200 ms/50 ms
configuration, so shuffling puts near-copies of the same window on both sides of
the split. How much that inflates the score depends on **model capacity**: a
high-capacity model (kNN, a deep net) can exploit the near-duplicates, while LDA
on ten features has nothing to memorise with and may show little or no gap. So
this row is a control on the *protocol-plus-model pair*, not a fixed correction
factor — and a small gap under LDA is not licence to drop it when a larger model
is adopted later. Report whatever it shows, including "no inflation here".

> **Scope.** This notebook establishes *accuracy* across protocols. What happens
> to **confidence calibration** under the same drift — the part that connects to
> the architecture paper — is `03_drift_fewshot`. The calibration and
> unsafe-assist columns appear here already because the harness computes them for
> free; read them as a preview, not as this notebook's claim.

## Setup

Needs scikit-learn (`pip install -e ".[ml]"`) on top of the phase-B environment.
Run with `py -3.11` (`docs/research/phase-b-plan.md` §10).

In [1]:
import csv
import json
import time
from pathlib import Path

import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

from reborn.data.evaluation import evaluate_splits, summarize
from reborn.data.features import FEATURE_NAMES, feature_matrix, standardize
from reborn.data.loaders import NinaproDB6Loader
from reborn.data.pipeline import PreprocessConfig, build_window_set
from reborn.data.splits import (
    cross_session_splits,
    cross_subject_splits,
    random_window_split,
    within_session_splits,
)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = REPO / "experiments" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

CONFIG = json.loads((REPO / "experiments" / "configs" / "ninapro_db6_qc.json").read_text())
QC_KWARGS = CONFIG["qc_kwargs"]
MONTAGE = tuple(CONFIG["channels"]["montage"])
PRE = CONFIG["preprocess"]

config = PreprocessConfig(
    target_sample_rate=PRE["target_sample_rate_hz"],
    bandpass_hz=tuple(PRE["bandpass_hz"]),
    notch_hz=PRE["notch_hz"],
    window_ms=PRE["window_ms"],
    stride_ms=PRE["stride_ms"],
    pure_windows=PRE["pure_windows"],
    qc_kwargs=QC_KWARGS,
)
print("config fingerprint:", config.fingerprint())
print("features:", FEATURE_NAMES)

config fingerprint: fed1e81a523e5d6e
features: ('rms', 'mav', 'zcr', 'wl', 'ssc')


## 1. Build the window set

One pass: load → resample → filter → window → QC gate. Windows that fail QC never
reach the classifier, which is the runtime order — the safety layer decides
whether EMG is trustworthy *before* anything acts on it.

**Cost.** Two subjects x 10 sessions is ~260k windows and takes minutes. The
cache below means you pay that once; delete the `.npz` to force a rebuild.
Cross-subject needs at least two subjects, so `s01` alone will skip that protocol.

In [2]:
from reborn.data.pipeline import load_window_set, save_window_set

SUBJECTS = ["s01", "s02"]
SESSIONS = None            # None = every session

cache = REPO / "data" / "cache" / f"db6_{'_'.join(SUBJECTS)}_{config.fingerprint()}.npz"
loader = NinaproDB6Loader(REPO / "data" / "ninapro_db6", channels=MONTAGE)

if cache.exists():
    window_set = load_window_set(cache)
    print(f"loaded cache: {cache.name}")
else:
    started = time.time()
    window_set = build_window_set(loader.load(subjects=SUBJECTS, sessions=SESSIONS), config)
    save_window_set(window_set, cache, config)
    print(f"built and cached in {time.time() - started:.0f} s")

sessions = sorted(map(str, set(window_set.session_ids)))
print(f"\n{window_set.n_windows} windows, {window_set.n_channels} channels")
print(f"subjects: {sorted(map(str, set(window_set.subject_ids)))}")
print(f"sessions: {len(sessions)}  {sessions}")
print(f"QC rejected {window_set.qc.rejection_rate:.2%} before this point")

# Drift needs elapsed time to accumulate. Loading only adjacent sessions produces
# a small cross-session gap for reasons that have nothing to do with the dataset.
if len(sessions) < 4:
    print(
        f"\nNOTE: only {len(sessions)} sessions loaded. DB6 spans 5 days x 2 sessions;"
        "\n      a cross-session result over this range understates drift by construction."
    )

loaded cache: db6_s01_s02_fed1e81a523e5d6e.npz

259745 windows, 2 channels
subjects: ['s01', 's02']
sessions: 10  ['d01_t01', 'd01_t02', 'd02_t01', 'd02_t02', 'd03_t01', 'd03_t02', 'd04_t01', 'd04_t02', 'd05_t01', 'd05_t02']
QC rejected 0.40% before this point


## 2. The task, and the honest reduction

Reborn's decision is binary — assist flexion or do not. DB6 labels seven grasps
plus rest. The binary task is derived as **rest vs. any movement**, which is the
closest honest mapping onto the elbow orthosis; the multi-class task is reported
alongside so the reduction is visible rather than assumed
(`docs/research/phase-b-plan.md` §3).

Class balance matters for reading the numbers below: if rest is a quarter of
windows, a model that always predicts "movement" already scores 75% accuracy.
That is why balanced accuracy is the headline column.

In [3]:
X_raw, feature_names = feature_matrix(window_set)
y_multi = window_set.labels
y_binary = window_set.binary_labels()

print(f"feature matrix: {X_raw.shape}  ({len(feature_names)} columns)")
print(f"columns: {feature_names}\n")

for name, y in (("multi-class", y_multi), ("binary (rest vs movement)", y_binary)):
    labels, counts = np.unique(y, return_counts=True)
    shares = "  ".join(f"{lab}:{n / len(y):.1%}" for lab, n in zip(labels, counts))
    print(f"{name:<28}{len(labels)} classes   {shares}")
    print(f"{'':<28}majority-class accuracy = {counts.max() / len(y):.1%}")

feature matrix: (259745, 10)  (10 columns)
columns: ['rms_ch0', 'rms_ch1', 'mav_ch0', 'mav_ch1', 'zcr_ch0', 'zcr_ch1', 'wl_ch0', 'wl_ch1', 'ssc_ch0', 'ssc_ch1']

multi-class                 8 classes   0:24.5%  1:11.4%  3:10.7%  4:11.0%  6:10.5%  9:11.0%  10:10.1%  11:10.7%
                            majority-class accuracy = 24.5%
binary (rest vs movement)   2 classes   0:24.5%  1:75.5%
                            majority-class accuracy = 75.5%


## 3. Models

LDA and calibrated logistic regression, both CPU-only. Deliberately simple: the
question this phase asks is what happens to *confidence* under drift, and a model
whose calibration can be reasoned about is worth more here than a point of
accuracy. A 1D CNN is warranted only if these turn out to be the limiting factor.

Two things the `fit_predict` closures below get right, and which are easy to get
wrong:

- **Standardisation is fitted on training rows only.** Scaling with statistics
  pooled over train and test leaks the per-session amplitude shift — which is
  precisely the drift under study — and flatters every cross-session number.
- **Confidence is the probability of the predicted class**, which is what
  `ConfidenceGate` consumes at runtime. LDA's posterior is used directly; nothing
  is rescaled to look better.

In [4]:
def make_fit_predict(estimator_factory):
    """Wrap an sklearn estimator as the harness's (X_tr, y_tr, X_te) -> (pred, conf)."""

    def fit_predict(X_train, y_train, X_test):
        X_train, X_test = standardize(X_train, X_test)
        model = estimator_factory().fit(X_train, y_train)
        proba = model.predict_proba(X_test)
        predictions = model.classes_[np.argmax(proba, axis=1)]
        confidence = np.max(proba, axis=1)
        return predictions, confidence

    return fit_predict


MODELS = {
    "lda": make_fit_predict(LinearDiscriminantAnalysis),
    "logreg": make_fit_predict(lambda: LogisticRegression(max_iter=2000)),
}
print("models:", list(MODELS))

models: ['lda', 'logreg']


## 4. The four protocols

Each protocol yields several splits (one per session, per subject, and so on).
They are evaluated individually and summarised with **spread, not just a mean** —
under cross-session the variation between held-out sessions *is* the drift, and
averaging it away discards the result.

In [5]:
protocols = {
    "within-session": within_session_splits(window_set),
    "cross-session": cross_session_splits(window_set, n_train_sessions=1),
    "cross-subject": cross_subject_splits(window_set),
    "random-shuffle": [random_window_split(window_set, seed=0)],
}

for name, splits in protocols.items():
    print(f"{name:<18}{len(splits):>3} splits")
    if not splits:
        print(f"{'':<18}    (none — needs more subjects or sessions than are loaded)")

within-session     20 splits
cross-session      18 splits
cross-subject       2 splits
random-shuffle      1 splits


In [6]:
# All four {task} x {model} combinations in one pass, so the table below is
# reproducible from a single execution rather than four manual runs.
COMBOS = [(task, model) for task in ("binary", "multi") for model in MODELS]
labels_for = {"binary": y_binary, "multi": y_multi}

results_by_combo = {}
started = time.time()
for task, model_name in COMBOS:
    y = labels_for[task]
    combo_results = []
    for name, splits in protocols.items():
        if not splits:
            continue
        combo_results.extend(evaluate_splits(X_raw, y, splits, MODELS[model_name], protocol=name))
    results_by_combo[(task, model_name)] = combo_results
    print(f"{task:<7}{model_name:<8}{len(combo_results):>3} splits evaluated")
print(f"\n({time.time() - started:.0f} s)")

binary lda      41 splits evaluated


binary logreg   41 splits evaluated


multi  lda      41 splits evaluated


multi  logreg   41 splits evaluated

(30 s)


## 5. The baseline table

Read the **within-session → cross-session** gap first: that is the drift, and it
is what notebook 03 builds on.

Two ways this can come out flat, and they are not the same finding:

- **Sessions too close together.** DB6 spans 5 days, two sessions per day. If
  `SESSIONS` covers only day 1 and day 2, there is barely any elapsed time for
  drift to accumulate, and a small gap says nothing about the dataset. Check §6
  before concluding anything.
- **Genuinely little drift across the full span.** That is a real result, and it
  is the one the stop rule is about.

> **Stop rule** (`docs/research/checklist.md`): if cross-session is
> indistinguishable from within-session *across the full session range*, there is
> no drift to study here and the paper's framing needs revisiting before
> continuing to notebook 03. Widening `SESSIONS` is the first thing to try, not
> the conclusion.

In [7]:
combos = list(results_by_combo)


def cell(combo, protocol, field):
    subset = [r for r in results_by_combo[combo] if r.protocol == protocol]
    return summarize(subset).get(field) if subset else None


summary_rows = []
for combo in combos:
    for name in protocols:
        subset = [r for r in results_by_combo[combo] if r.protocol == name]
        if subset:
            summary_rows.append(
                {"task": combo[0], "model": combo[1], "protocol": name, **summarize(subset)}
            )

print("balanced accuracy  —  protocol x (task/model)")
header = f"{'protocol':<16}" + "".join(f"{t + '/' + m:>14}" for t, m in combos)
print(header)
print("-" * len(header))
for name in protocols:
    if not protocols[name]:
        continue
    row = f"{name:<16}"
    for combo in combos:
        v = cell(combo, name, "balanced_accuracy_mean")
        row += f"{v:>14.3f}" if v is not None else f"{'-':>14}"
    print(row)

print("\nwithin-session -> cross-session  (the drift, per combo):")
for combo in combos:
    w = cell(combo, "within-session", "balanced_accuracy_mean")
    x = cell(combo, "cross-session", "balanced_accuracy_mean")
    we = cell(combo, "within-session", "ece_mean")
    xe = cell(combo, "cross-session", "ece_mean")
    if None in (w, x, we, xe):
        continue
    print(
        f"  {combo[0]:<7}{combo[1]:<8} bal.acc {w:.3f} -> {x:.3f} ({x - w:+.3f})   "
        f"ECE {we:.3f} -> {xe:.3f} (x{xe / we:.1f})"
    )

sh = ("binary", "lda")
if sh in results_by_combo:
    s_ba = cell(sh, "random-shuffle", "balanced_accuracy_mean")
    x_ba = cell(sh, "cross-session", "balanced_accuracy_mean")
    if s_ba is not None and x_ba is not None:
        print(f"\nrandom-shuffle inflation over cross-session ({sh[0]}/{sh[1]}): {s_ba - x_ba:+.3f}")

balanced accuracy  —  protocol x (task/model)
protocol            binary/lda binary/logreg     multi/lda  multi/logreg
------------------------------------------------------------------------
within-session           0.917         0.949         0.468         0.477
cross-session            0.896         0.929         0.359         0.376
cross-subject            0.912         0.945         0.273         0.284
random-shuffle           0.901         0.951         0.343         0.361

within-session -> cross-session  (the drift, per combo):
  binary lda      bal.acc 0.917 -> 0.896 (-0.021)   ECE 0.016 -> 0.027 (x1.7)
  binary logreg   bal.acc 0.949 -> 0.929 (-0.020)   ECE 0.012 -> 0.028 (x2.4)
  multi  lda      bal.acc 0.468 -> 0.359 (-0.109)   ECE 0.065 -> 0.164 (x2.5)
  multi  logreg   bal.acc 0.477 -> 0.376 (-0.102)   ECE 0.030 -> 0.109 (x3.6)

random-shuffle inflation over cross-session (binary/lda): +0.005


## 6. Degradation against elapsed sessions

Cross-session splits carry `sessions_elapsed` — how far the test session sits
from the training one. If drift is real and cumulative, performance should fall
as this grows; if it is a fixed per-session offset, it should drop once and then
sit flat. Those are different physical stories and they call for different
personalization strategies in notebook 03, so the shape matters more than the
mean.

In [8]:
focus = ("binary", "lda")  # Reborn's actual decision, simplest model
by_elapsed = {}
for result in results_by_combo.get(focus, []):
    if result.protocol != "cross-session":
        continue
    by_elapsed.setdefault(result.meta.get("sessions_elapsed"), []).append(result)

print(f"cross-session degradation vs. elapsed sessions  —  {focus[0]}/{focus[1]}")
if by_elapsed:
    print(f"{'elapsed':>8}{'splits':>8}{'bal.acc':>10}{'ECE':>8}{'unsafe':>9}")
    print("-" * 43)
    for elapsed in sorted(k for k in by_elapsed if k is not None):
        subset = by_elapsed[elapsed]
        stats = summarize(subset)
        print(
            f"{elapsed:>8}{len(subset):>8}{stats['balanced_accuracy_mean']:>10.3f}"
            f"{stats['ece_mean']:>8.3f}{stats['unsafe_assist_rate_mean']:>9.3f}"
        )
else:
    print("no cross-session splits — load more sessions")

cross-session degradation vs. elapsed sessions  —  binary/lda
 elapsed  splits   bal.acc     ECE   unsafe
-------------------------------------------
       1       2     0.906   0.016    0.029
       2       2     0.924   0.023    0.014
       3       2     0.756   0.097    0.048
       4       2     0.932   0.018    0.013
       5       2     0.918   0.019    0.018
       6       2     0.926   0.023    0.017
       7       2     0.886   0.014    0.032
       8       2     0.909   0.019    0.015
       9       2     0.909   0.015    0.026


## 7. Artifacts

Per-split rows and the protocol summary, both keyed by the preprocessing
fingerprint. Figures come from a separate script reading these CSVs, so no number
in the paper depends on a live kernel (`docs/research/phase-b-plan.md` §8).

In [9]:
def write_csv(path, rows):
    if not rows:
        print(f"skipped {path.name}: nothing to write")
        return
    fields = sorted({key for row in rows for key in row})
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    print(f"wrote {path.relative_to(REPO)}  ({len(rows)} rows)")


fp = config.fingerprint()
for (task, model_name), results in results_by_combo.items():
    write_csv(RESULTS / f"nb02_splits_{task}_{model_name}_{fp}.csv", [r.as_row() for r in results])
write_csv(RESULTS / f"nb02_summary_{fp}.csv", summary_rows)

(RESULTS / f"nb02_run_{fp}.json").write_text(
    json.dumps(
        {
            "config_fingerprint": fp,
            "dataset": "ninapro_db6",
            "subjects": SUBJECTS,
            "sessions": SESSIONS or "all",
            "montage_raw_columns": list(MONTAGE),
            "combos": [f"{t}_{m}" for t, m in combos],
            "features": list(FEATURE_NAMES),
            "n_windows": window_set.n_windows,
            "qc_rejection_rate": window_set.qc.rejection_rate,
        },
        indent=2,
    )
)
print(f"wrote experiments/results/nb02_run_{fp}.json")

wrote experiments\results\nb02_splits_binary_lda_fed1e81a523e5d6e.csv  (41 rows)
wrote experiments\results\nb02_splits_binary_logreg_fed1e81a523e5d6e.csv  (41 rows)
wrote experiments\results\nb02_splits_multi_lda_fed1e81a523e5d6e.csv  (41 rows)
wrote experiments\results\nb02_splits_multi_logreg_fed1e81a523e5d6e.csv  (41 rows)
wrote experiments\results\nb02_summary_fed1e81a523e5d6e.csv  (16 rows)
wrote experiments/results/nb02_run_fed1e81a523e5d6e.json


## 8. Reading & next steps

### Notes from this run

_DB6, subjects s01–s02, all 10 sessions, 259 745 windows, montage (0, 10). QC
rejected 0.40%. Config fingerprint `fed1e81a523e5d6e`. Four runs: {binary,
multi} x {LDA, logreg}._

**Balanced accuracy**

| protocol | binary LDA | binary logreg | multi LDA | multi logreg |
|---|---|---|---|---|
| within-session | 0.917 | 0.949 | 0.468 | 0.477 |
| cross-session | 0.896 | 0.929 | 0.359 | 0.376 |
| cross-subject | 0.912 | 0.945 | 0.273 | 0.284 |
| random-shuffle | 0.901 | 0.951 | 0.343 | 0.361 |

- **The binary task barely drifts; the multi-class task drifts a lot.** Binary
  loses 0.021 (LDA) / 0.020 (logreg) balanced accuracy across sessions;
  multi-class loses **0.109 / 0.102**. Rest-vs-movement is close to a decision on
  signal energy, and energy is what survives an electrode being replaced the next
  day. *Which* gesture is being made does not survive it. **This matters for
  Reborn specifically**: its decision is the binary one, so the drift threat to
  *this* system is milder than the gesture-recognition literature implies — and
  the paper should say so rather than borrowing that literature's alarm.

- **Calibration degrades faster than accuracy does.** This is the finding the
  phase connects to the architecture paper. Binary ECE goes 0.016 → 0.027 (LDA)
  and 0.012 → 0.028 (logreg) across sessions; multi-class 0.065 → 0.164 and
  0.030 → 0.109. In every one of the four runs ECE roughly **doubles to
  triples**, while accuracy falls by a few points at most. The model does not
  become much worse — it becomes **much more sure than it deserves to be**, which
  is exactly the condition under which the confidence gate opens on predictions
  it should refuse.

- **The drift is episodic, not cumulative (§6).** Degradation does not grow with
  elapsed sessions: binary LDA reads 0.906, 0.924, **0.756**, 0.932, 0.918,
  0.926, 0.886, 0.909, 0.909 for 1–9 sessions elapsed. One session
  (`sessions_elapsed = 3`) collapses; the rest sit flat. At that session ECE
  jumps to 0.097 (5x its neighbours) and unsafe-assist to 0.048 (~3x). **Design
  implication, and it is the main one:** a personalization strategy that
  recalibrates on a fixed schedule addresses a drift shape this dataset does not
  show. What the data asks for is *event detection* — noticing that this session
  is anomalous — which is the same requirement notebook 01 reached from the QC
  side. Two independent routes to the same conclusion.

- **Cross-subject is not the worst case for the binary task.** It scores 0.912 /
  0.945 — matching or beating cross-session. For a binary energy decision,
  another person's data generalises as well as the same person's on another day.
  For multi-class it behaves as expected (0.273 / 0.284, clearly worst). So
  few-shot personalization has much less headroom on the binary task than the
  roadmap assumed, and notebook 03 should test that assumption before optimising
  against it.

- **The random-shuffle control showed no inflation, as the model predicts.**
  0.901 vs 0.896 (LDA), 0.951 vs 0.929 (logreg) — within noise of cross-session,
  and *negative* for multi-class. As §1 says, the overlapping-window inflation is
  capacity-dependent and LDA has nothing to memorise with. The row stays: it must
  be re-run before any higher-capacity model is trusted.

- **Two channels cap the multi-class task.** 0.47 balanced accuracy over 8
  classes (chance 0.125) is well above chance but poor in absolute terms. That is
  a montage limit, not a modelling failure — two electrodes cannot separate seven
  hand grasps. It does not affect the binary conclusions, which are what Reborn
  needs, but any multi-class number here should be read as a floor and never
  compared against 14-channel literature results.

- **Logistic regression beats LDA everywhere**, by ~0.03 balanced accuracy and
  with better calibration (binary cross-session ECE 0.028 vs 0.027 at higher
  accuracy). Not enough to motivate a CNN.

### Methodological correction made during this run

The within-session protocol originally split **temporally**, which is wrong for
Ninapro: DB6 records all 12 repetitions of one grasp before moving to the next,
so each class occupies its own contiguous ~14% of the recording. A 70/30 temporal
cut trained on classes {0,1,3,4,6,9} and tested on {0,9,10,11} — two classes
never trained, four never tested. The symptom was that the "ceiling" sat *below*
cross-session (multi-class 0.363 vs 0.359, and accuracy 0.291 vs 0.429).
`within_session_splits` now splits **by repetition** when the loader provides
repetition numbers, which is also the Ninapro convention; the multi-class ceiling
moved to 0.468 and behaves like a ceiling. The temporal path remains as a
fallback and flags itself in `split.meta`.

### Next

1. Notebook 03 asks the question this one raises: given that ECE doubles while
   accuracy holds, **how far does the unsafe-assist rate move as the gate
   threshold sweeps**, and does few-shot calibration restore calibration faster
   than it restores accuracy?
2. Investigate `sessions_elapsed = 3` directly. One session drives the entire
   cross-session degradation; naming what happened in it is worth more than
   another model.
3. The stop rule does **not** fire, but it fires for the binary task in the form
   the roadmap expected: there is no gradual drift to personalise against. Notebook
   03's framing should be episodic degradation, not slow drift.